# Text Splitting Techniques

This notebook compares common splitters used before embedding:
- CharacterTextSplitter
- RecursiveCharacterTextSplitter
- TokenTextSplitter

It also explains why some chunks may exceed a configured limit.

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter,
)


PROJECT_ROOT = Path.cwd().resolve().parents[1]
DATA_FILE = PROJECT_ROOT / "data" / "sample_dataset.txt"

if not DATA_FILE.exists():
    raise FileNotFoundError(f"Missing text file: {DATA_FILE}")

loader = TextLoader(str(DATA_FILE), encoding="utf-8")
documents = loader.load()
sample_text = documents[0].page_content

print("Loaded chars:", len(sample_text))

In [ ]:
CHUNK_SIZE = 800
CHUNK_OVERLAP = 120

if CHUNK_SIZE <= 0:
    raise ValueError("CHUNK_SIZE must be > 0")
if CHUNK_OVERLAP < 0 or CHUNK_OVERLAP >= CHUNK_SIZE:
    raise ValueError("CHUNK_OVERLAP must be >= 0 and < CHUNK_SIZE")


def preview(name: str, chunks: list[str]) -> None:
    print(f"\n{name}")
    print("chunks:", len(chunks))
    print("first chunk chars:", len(chunks[0]) if chunks else 0)
    if chunks:
        print("preview:", chunks[0][:180], "...")

In [ ]:
char_splitter = CharacterTextSplitter(
    separator="\n",
    is_separator_regex=False,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
char_chunks = char_splitter.split_text(sample_text)
preview("CharacterTextSplitter", char_chunks)

recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""],
    keep_separator=True,
    is_separator_regex=False,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
recursive_chunks = recursive_splitter.split_text(sample_text)
preview("RecursiveCharacterTextSplitter", recursive_chunks)

token_splitter = TokenTextSplitter(
    encoding_name="cl100k_base",
    chunk_size=200,
    chunk_overlap=30,
)
token_chunks = token_splitter.split_text(sample_text)
preview("TokenTextSplitter", token_chunks)

## Why "chunk longer than configured size" can happen

If a splitter uses a separator (for example `\n`) and a single segment between separators is already larger than `chunk_size`, it may emit an oversized chunk and print a warning.

Mitigation options:
- Use `RecursiveCharacterTextSplitter` with fallback separators
- Increase `chunk_size`
- Pre-clean text to add better breakpoints